In [ ]:
!pip install -qu oci==2.154.0

In [ ]:
import oci
print(oci.__version__)

In [ ]:
CONFIG_PROFILE = "DEFAULT"
configmk = oci.config.from_file('~/.oci/config', CONFIG_PROFILE)

# Testing connection for Chat Response - MK account

generative_ai_inference_client=oci.generative_ai_inference.GenerativeAiInferenceClient(
    config=configmk,    
    service_endpoint="https://inference.generativeai.us-chicago-1.oci.oraclecloud.com",
    retry_stratergy=oci.retry.NoneRetryStrategy(),
    timeout=(10,240)
)

chat_detail = oci.generative_ai_inference.models.ChatDetails()
chat_request=oci.generative_ai_inference.models.CohereChatRequest()
chat_request.message="How does a telescope work"
chat_request.max_tokens=600
chat_request.temperature=1
chat_request.frequency_penalty=0
chat_request.top_p=0.75
chat_request.top_k=0

chat_detail.serving_mode= oci.generative_ai_inference.models.OnDemandServingMode(
    model_id="cohere.command-r-08-2024")
chat_detail.chat_request=chat_request
chat_detail.compartment_id="ocid1.tenancy.oc1..aaaaaaaajknwd2e6sytwavwr7s4bhadiadizv5z6n5xqrrseyhxkzz6vvega"

chat_response = generative_ai_inference_client.chat(chat_detail)

print("***************ChatResult***********")
print(vars(chat_response))

In [ ]:
# Single Tool

from typing import Dict
from oci.addons.adk import Agent, AgentClient, tool

@tool
def get_weather(location: str) -> Dict[str, str]:
    """
    Get the weather for a given location.

    Args:
      location(str): The location for which weather is queried
    """
    return {"location": location, "temperature": 72, "unit": "F"}


def main():

    
    client = AgentClient(
        auth_type="api_key",
        profile="DEFAULT",
        region="us-chicago-1",
    )


    agent = Agent(
        client=client,
        agent_endpoint_id="ocid1.genaiagentendpoint.oc1.us-chicago-1.amaaaaaa2fm4ibaamar7py3zzjo2hfd5onda3sa7p47daadpngpzkios6yfq",
        instructions="You perform weather queries using tools.",
        tools=[get_weather]
    )


    agent.setup()


    input = "Is it cold in Seattle?"
    response = agent.run(input)

    # Print the response
    response.pretty_print()

if __name__ == "__main__":
    main()